# Running all five released ONNX models on one demo crop

This notebook loads every ONNX model bundled in `onnx_models/` and runs each
one on a single chromosome-pair crop, as a quick usage guide for a reader who
just cloned the repository.

Models covered:

| Adapter class | ONNX file |
|---|---|
| `SuSyOnnxAdapter` | `massnet_susy.onnx` |
| `SuReOnnxAdapter` | `massnet_sure.onnx` |
| `SuSyToSuReOnnxAdapter` | `massnet_susy_to_sure.onnx` |
| `SuSyToSeDANNOnnxAdapter` | `massnet_susy_to_sedann.onnx` |
| `YoloOverlapTouchOnnxAdapter` | `yolo26n_seg_overlap_touch_core2.onnx` |

The demo crop is loaded from your own local image. Its path is supplied at
runtime and is not stored in this notebook's source. Saved outputs can embed
the image and predictions: clear all outputs before committing or sharing.


In [ ]:
! pip install -r requirements-notebook.txt

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))  # run from onnx_models/

from adapters import (
    SuSyOnnxAdapter,
    SuReOnnxAdapter,
    SuSyToSuReOnnxAdapter,
    SuSyToSeDANNOnnxAdapter,
)
from yolo_overlap_touch_adapter import YoloOverlapTouchOnnxAdapter, derive_pairwise_overlap

## 1. Load a demo crop

Set the `CHROMAADAPT_DEMO_PATH` environment variable before launching Jupyter,
or enter your local image path in the hidden prompt below. Keep private paths
out of notebook cells. Clear all outputs before committing or sharing, since
rendered plots contain your image data.


In [ ]:
import os
from getpass import getpass

demo_path = os.environ.get("CHROMAADAPT_DEMO_PATH") or getpass("Local demo image path (hidden): ")
if not Path(demo_path).is_file():
    raise FileNotFoundError("Demo image not found; provide a valid local image path.")
demo_bgr = cv2.imread(demo_path)
if demo_bgr is None:
    raise ValueError("Could not decode the demo image; choose a supported image file.")

plt.figure(figsize=(3, 3))
plt.imshow(cv2.cvtColor(demo_bgr, cv2.COLOR_BGR2RGB))
plt.title("Demo crop")
plt.axis("off")
plt.show()


## 2. Load all five adapters

Each adapter loads its matching ONNX file from this directory by default and
runs on CPU (`CPUExecutionProvider`).

In [ ]:
models = {
    "SuSy": SuSyOnnxAdapter(),
    "SuRe": SuReOnnxAdapter(),
    "SuSy to SuRe": SuSyToSuReOnnxAdapter(),
    "SuSy to SeDANN": SuSyToSeDANNOnnxAdapter(),
    "YOLO overlap-touch": YoloOverlapTouchOnnxAdapter(),
}
list(models)

## 3. Run inference with every model on the same crop

The four MASS-Net regimes share one call shape (`predict_image` -> dense
3-class chromosome labels + a binary overlap mask, all at a fixed 224x224).
The YOLO adapter has a different output shape: a variable number of
(box, score, mask) instances at the crop's own resolution.

In [ ]:
import time

predictions = {}
for name, model in models.items():
    start = time.perf_counter()
    predictions[name] = model.predict_image(demo_bgr)
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"{name:20s} ran in {elapsed_ms:6.1f} ms")

## 4. Visualize every model's output side by side

In [ ]:
CHROMOSOME_PALETTE = np.array([[0, 0, 0], [220, 60, 60], [60, 140, 220]], dtype=np.uint8)  # bg/inst1/inst2, RGB
OVERLAP_COLOR = np.array([255, 220, 0], dtype=np.uint8)  # yellow, matches the MASS-Net panels

fig, axes = plt.subplots(1, len(models) + 1, figsize=(4 * (len(models) + 1), 4))

axes[0].imshow(cv2.cvtColor(demo_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("input crop")
axes[0].axis("off")

for ax, (name, pred) in zip(axes[1:], predictions.items()):
    if hasattr(pred, "chromosome_labels"):  # MASS-Net family
        labels = pred.chromosome_labels[0]
        labels_resized = cv2.resize(labels, (demo_bgr.shape[1], demo_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
        overlap_resized = cv2.resize(pred.overlap_masks[0], (demo_bgr.shape[1], demo_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
        rgb = CHROMOSOME_PALETTE[labels_resized]
        rgb[overlap_resized.astype(bool)] = OVERLAP_COLOR
    else:  # YoloOverlapTouchPrediction -- no dedicated overlap head, so the
           # overlap region (if any) is derived from the two instance masks'
           # intersection, purely for this visualization.
        rgb = cv2.cvtColor(demo_bgr, cv2.COLOR_BGR2RGB).copy()
        for i, mask in enumerate(pred.masks[:2]):
            rgb[mask] = CHROMOSOME_PALETTE[i + 1]
        overlap = derive_pairwise_overlap(pred)
        if overlap is not None:
            rgb[overlap] = OVERLAP_COLOR
        for box in pred.boxes_xyxy:
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(rgb, (x1, y1), (x2, y2), (120, 120, 120), 1)
    ax.imshow(rgb)
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

Reading the panels:

- **MASS-Net regimes** (red/blue): one color per predicted chromosome
  instance, yellow = pixels predicted as the overlap region from the
  model's own dedicated overlap head. All four share the same 224x224
  output resolution regardless of the crop's native size.
- **YOLO overlap-touch**: one colored mask per detected instance (red/blue,
  same palette, up to two), grey boxes mark each detection. This model has
  no dedicated overlap head -- the yellow region is *derived*, by this
  notebook, as the pixel intersection of the two instance masks
  (`derive_pairwise_overlap`), not predicted directly by the network.

After running the notebook, these panels show predictions on your supplied
crop. See the paper and `onnx_models/README.md`'s validation tables for
the full accuracy numbers this single example cannot represent on its own.

## 5. Using your own image

To use a different crop, update `CHROMAADAPT_DEMO_PATH` or enter a new path at
the hidden prompt, then re-run the loading, inference, and visualization cells.

Or use each adapter's `predict_path(...)`, which loads and preprocesses the
image for you and returns `(original_bgr_image, prediction)`:

```python
image, prediction = models["SuSy to SuRe"].predict_path(demo_path)
```

Before committing or sharing, use **Restart Kernel and Clear All Outputs** and
save the notebook to remove rendered images, predictions, and local tracebacks.
